# Lab 4 — Modern Classification Algorithms (Student Template)


---

## IMPORTANT
1. **Make a copy** (File → Save a copy in Drive) before starting.
2. Do **not** modify cells marked **PROVIDED — DO NOT MODIFY**.
3. Write your code only in the **Student Task** code cells.


## 1) Imports (PROVIDED — DO NOT MODIFY)

In [10]:
# ======================================================
# PROVIDED CODE — DO NOT MODIFY
# ======================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


## 2) Dataset Generation (PROVIDED — DO NOT MODIFY)

In [11]:
# ======================================================
# PROVIDED CODE — DO NOT MODIFY
# ======================================================
# Each row represents one network session.
# This synthetic dataset is generated only for learning purposes.

np.random.seed(42)
n_samples = 500

# Number of failed login attempts in the session
failed_logins = np.random.poisson(lam=2, size=n_samples)

# Amount of data transferred during the session (in MB)
data_volume_mb = np.random.normal(loc=300, scale=120, size=n_samples).clip(50, 1000)

# Whether the session occurred at an unusual time (1 = yes, 0 = no)
unusual_time = np.random.binomial(1, 0.25, size=n_samples)

# Days since the system was last patched
patch_age_days = np.random.randint(0, 365, size=n_samples)

# Whether an admin account was used in the session
admin_login = np.random.binomial(1, 0.15, size=n_samples)

# Cyber risk score (0–100). Higher means higher risk.
risk_score = (
    5 * failed_logins +
    0.04 * data_volume_mb +
    15 * unusual_time +
    0.03 * patch_age_days +
    20 * admin_login +
    0.5 * failed_logins**2 +                 # nonlinear effect
    np.random.normal(0, 5, size=n_samples)   # noise
)

risk_score = np.clip(risk_score, 0, 100)

df = pd.DataFrame({
    "failed_logins": failed_logins,
    "data_volume_mb": data_volume_mb,
    "unusual_time": unusual_time,
    "patch_age_days": patch_age_days,
    "admin_login": admin_login,
    "risk_score": risk_score
})

df.head()


,failed_logins,data_volume_mb,unusual_time,patch_age_days,admin_login,risk_score
0,4,491.340608,0,216,0,54.978987
1,1,198.364638,0,119,0,15.001039
2,3,181.032918,0,174,0,21.995223
3,3,50.000000,0,22,0,11.908258
4,1,223.324590,0,212,1,43.741031


## 3) Binary Labels + Train/Test Split (PROVIDED — DO NOT MODIFY)

In [12]:
# ======================================================
# PROVIDED CODE — DO NOT MODIFY
# ======================================================

# High Risk (1): risk_score >= 70
# Low Risk  (0): risk_score < 70
X = df.drop(columns=["risk_score"])
y = (df["risk_score"] >= 70).astype(int)

# Stratify keeps the class proportions similar in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train classes:")
print(y_train.value_counts())
print("\nTest classes:")
print(y_test.value_counts())


Train classes:
risk_score
0    387
1     13
Name: count, dtype: int64

Test classes:
risk_score
0    97
1     3
Name: count, dtype: int64


## 4) Evaluation Helper (PROVIDED — DO NOT MODIFY)

In [13]:
# ======================================================
# PROVIDED CODE — DO NOT MODIFY
# ======================================================
# Prints confusion matrix + common classification metrics.

def clf_report_simple(y_true, y_pred, model_name):
    print(model_name)
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}")
    print(f"F1-score : {f1_score(y_true, y_pred, zero_division=0):.3f}")
    print("-" * 40)


# ✅ Student Tasks

Write your code in the empty code cells under each task.

**Tip:** Evaluate using `clf_report_simple(y_test, y_pred, "Model Name")`.


## Task 1 — Decision Tree

**Goal:** Train a decision tree classifier.

**What to do:** Train → Predict → Evaluate.

**What to observe:** Does it overfit? How is recall?


In [14]:
# Student code here

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

# Evaluate
clf_report_simple(y_test, y_pred_dt, "DECISION TREE")

# Check for overfitting
print(f"Tree Depth: {dt_model.get_depth()}")
print(f"Training Accuracy: {dt_model.score(X_train, y_train):.3f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_dt):.3f}")


DECISION TREE
Confusion Matrix:
[[94  3]
 [ 3  0]]
Accuracy : 0.940
Precision: 0.000
Recall   : 0.000
F1-score : 0.000
----------------------------------------
Tree Depth: 6
Training Accuracy: 1.000
Test Accuracy: 0.940


## Task 2 — Random Forest

**Goal:** Use many trees to improve stability.

**What to do:** Train → Predict → Evaluate.

**What to observe:** Compare to the single tree.


In [15]:
# Student code here

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
clf_report_simple(y_test, y_pred_rf, "RANDOM FOREST")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print("\nFeature Importance:")
print(feature_importance)



RANDOM FOREST
Confusion Matrix:
[[97  0]
 [ 3  0]]
Accuracy : 0.970
Precision: 0.000
Recall   : 0.000
F1-score : 0.000
----------------------------------------

Feature Importance:
          Feature  Importance
0   failed_logins    0.313565
3  patch_age_days    0.246454
1  data_volume_mb    0.238526
4     admin_login    0.150122
2    unusual_time    0.051334


## Task 3 — KNN

**Goal:** Distance-based classification.

**What to do:** Use a Pipeline with StandardScaler + KNN.

**What to observe:** How scaling affects results.


In [16]:
# Student code here


knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Train
knn_pipeline.fit(X_train, y_train)

# Predict
y_pred_knn = knn_pipeline.predict(X_test)

# Evaluate
clf_report_simple(y_test, y_pred_knn, "KNN (k=5)")

# Try different k values
print("\nTrying different k values:")
for k in [3, 5, 7, 9, 11]:
    knn_test = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    knn_test.fit(X_train, y_train)
    y_pred = knn_test.predict(X_test)
    print(f"k={k}: Accuracy={accuracy_score(y_test, y_pred):.3f}, Recall={recall_score(y_test, y_pred):.3f}")



KNN (k=5)
Confusion Matrix:
[[97  0]
 [ 2  1]]
Accuracy : 0.980
Precision: 1.000
Recall   : 0.333
F1-score : 0.500
----------------------------------------

Trying different k values:
k=3: Accuracy=0.980, Recall=0.333
k=5: Accuracy=0.980, Recall=0.333
k=7: Accuracy=0.980, Recall=0.333
k=9: Accuracy=0.970, Recall=0.000
k=11: Accuracy=0.970, Recall=0.000


## Task 4 — SVM

**Goal:** Margin-based classification.

**What to do:** Use a Pipeline with StandardScaler + SVC.

**What to observe:** Compare precision/recall to other models.


In [17]:
# Student code here

svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

# Train
svm_pipeline.fit(X_train, y_train)

# Predict
y_pred_svm = svm_pipeline.predict(X_test)

# Evaluate
clf_report_simple(y_test, y_pred_svm, "SVM (RBF kernel)")

# Try different kernels
print("\nTrying different kernels:")
for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
    svm_test = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, random_state=42))
    ])
    svm_test.fit(X_train, y_train)
    y_pred = svm_test.predict(X_test)
    print(f"{kernel:8s}: Accuracy={accuracy_score(y_test, y_pred):.3f}, Recall={recall_score(y_test, y_pred):.3f}")


SVM (RBF kernel)
Confusion Matrix:
[[97  0]
 [ 2  1]]
Accuracy : 0.980
Precision: 1.000
Recall   : 0.333
F1-score : 0.500
----------------------------------------

Trying different kernels:
linear  : Accuracy=0.990, Recall=1.000
poly    : Accuracy=0.990, Recall=0.667
rbf     : Accuracy=0.980, Recall=0.333
sigmoid : Accuracy=0.960, Recall=0.333


## Task 5 — Compare Models

Create a small DataFrame with Accuracy, Precision, Recall, F1 for each model.


In [18]:
# Student code here

comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'KNN (k=5)', 'SVM (RBF)'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_svm)
    ],
    'Precision': [
        precision_score(y_test, y_pred_dt, zero_division=0),
        precision_score(y_test, y_pred_rf, zero_division=0),
        precision_score(y_test, y_pred_knn, zero_division=0),
        precision_score(y_test, y_pred_svm, zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, y_pred_dt, zero_division=0),
        recall_score(y_test, y_pred_rf, zero_division=0),
        recall_score(y_test, y_pred_knn, zero_division=0),
        recall_score(y_test, y_pred_svm, zero_division=0)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_dt, zero_division=0),
        f1_score(y_test, y_pred_rf, zero_division=0),
        f1_score(y_test, y_pred_knn, zero_division=0),
        f1_score(y_test, y_pred_svm, zero_division=0)
    ]
})

print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
print(comparison.round(3).to_string(index=False))

# Find best model for recall
best_recall_model = comparison.loc[comparison['Recall'].idxmax(), 'Model']
best_recall = comparison['Recall'].max()
print(f"\n Best Recall: {best_recall_model} ({best_recall:.3f})")


MODEL COMPARISON
        Model  Accuracy  Precision  Recall  F1-Score
Decision Tree      0.94        0.0   0.000       0.0
Random Forest      0.97        0.0   0.000       0.0
    KNN (k=5)      0.98        1.0   0.333       0.5
    SVM (RBF)      0.98        1.0   0.333       0.5

 Best Recall: KNN (k=5) (0.333)


## Reflection Questions

1. Which model achieved the highest **recall**? Why is recall important in cyber risk?

*   SVM with linear kernel had the highest recall(100%). it caught all 3 high risk sessions.Recall matters because missing an attack can lead to data breaches and huge losses. Better to check false alarms than miss a real threat.


2. Which model seems most likely to overfit? (2–3 lines)

*  Decision Tree. It got 100% on training data but only 94% on test
   data. It also missed all attacks (0% recall). This means it just
   memorized the training data instead of learning real patterns.


3. Which model would you deploy and why?


*  SVM with linear kernel. It caught every attack (100% recall) and
   had 99% accuracy. In security, catching all attacks is the main
   goal. KNN and RBF SVM missed 2 out of 3 attacks, which is too risky.



## Final Checklist

- [☓] All tasks completed
- [☓] All cells run without errors
- [☓] Outputs are visible
- [☓] Renamed to `Lab2_StudentName.ipynb`
- [☓] Downloaded and submitted the `.ipynb`
